In [49]:
import os
import sys
from pathlib import Path

os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

In [50]:
PROJECT_ROOT = Path.cwd()


if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))

from models.als import ALSCandidateModel
from src import temporal_train_validation_test_split
from src import evaluate_recall_at_k
from src import evaluate_ndcg_at_k, oracle_ndcg_at_k

In [51]:
import gc
import time

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from models.als import ALSCandidateModel

### Загрузка и разделение данных

In [52]:
interactions = pd.read_csv(
    PROJECT_ROOT / "data" / "interactions.csv"
)

print(f"Количество взаимодействий: {len(interactions):,}")
display(interactions.head())

Количество взаимодействий: 1,594,787


,user_id,item_id,last_watch_dt,total_dur,watched_pct
0,176549,9506,2021-05-11,4250.0,72.0
1,699317,1659,2021-05-29,8317.0,100.0
2,656683,7107,2021-05-09,10.0,0.0
3,864613,7638,2021-07-05,14483.0,100.0
4,964868,9506,2021-04-30,6725.0,100.0


In [53]:
TRAIN_CUTOFF = "2021-07-01"
VALIDATION_CUTOFF = "2021-08-01"

train, validation, test = temporal_train_validation_test_split(
    interactions=interactions,
    train_cutoff=TRAIN_CUTOFF,
    validation_cutoff=VALIDATION_CUTOFF,
    date_col="last_watch_dt",
    drop_invalid_dates=True,
)

print(f"Train:      {len(train):,}")
print(f"Validation: {len(validation):,}")
print(f"Test:       {len(test):,}")

Train:      797,515
Validation: 427,045
Test:       370,226


/var/folders/ds/jdj_sy6s67ncz787qzz3xpc00000gn/T/ipykernel_20936/557796774.py:4: UserWarning: Dropped 1 rows with missing or invalid 'last_watch_dt'
  train, validation, test = temporal_train_validation_test_split(


### Сэмплирование тёплых пользователей

In [ ]:
SAMPLE_SIZE = 10_000
SAMPLE_RANDOM_STATE = 42

warm_user_ids = train["user_id"].drop_duplicates().to_numpy()

rng = np.random.default_rng(SAMPLE_RANDOM_STATE)
sampled_user_ids = np.sort(
    rng.choice(
        warm_user_ids,
        size=min(SAMPLE_SIZE, len(warm_user_ids)),
        replace=False,
    )
)

train = (
    train[train["user_id"].isin(sampled_user_ids)]
    .copy()
    .reset_index(drop=True)
)
validation = (
    validation[validation["user_id"].isin(sampled_user_ids)]
    .copy()
    .reset_index(drop=True)
)
test = (
    test[test["user_id"].isin(sampled_user_ids)]
    .copy()
    .reset_index(drop=True)
)

print(f"Выбрано пользователей: {len(sampled_user_ids):,}")
print(
    f"Train:      {len(train):,} взаимодействий, "
    f"{train['user_id'].nunique():,} пользователей"
)
print(
    f"Validation: {len(validation):,} взаимодействий, "
    f"{validation['user_id'].nunique():,} пользователей"
)
print(
    f"Test:       {len(test):,} взаимодействий, "
    f"{test['user_id'].nunique():,} пользователей"
)

Выбрано пользователей: 50,000
Train:      124,343 взаимодействий, 50,000 пользователей
Validation: 28,185 взаимодействий, 11,615 пользователей
Test:       15,069 взаимодействий, 6,995 пользователей


# Кандидатогенерация

## ALS

In [55]:
K = 200
BATCH_SIZE = 2_000
RANDOM_STATE = 42

train_history = (
    train[["user_id", "item_id"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

validation_targets = (
    validation[["user_id", "item_id"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

test_targets = (
    test[["user_id", "item_id"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

### Обучение и подбор гиперпараметров

In [56]:
PARAM_GRID = [
    {
        "factors": 32,
        "regularization": 0.05,
        "alpha": 10.0,
        "iterations": 20,
    },
    # {
    #     "factors": 64,
    #     "regularization": 0.05,
    #     "alpha": 10.0,
    #     "iterations": 20,
    # },
    # {
    #     "factors": 64,
    #     "regularization": 0.05,
    #     "alpha": 20.0,
    #     "iterations": 20,
    # },
    # {
    #     "factors": 64,
    #     "regularization": 0.10,
    #     "alpha": 20.0,
    #     "iterations": 20,
    # },
    # {
    #     "factors": 128,
    #     "regularization": 0.05,
    #     "alpha": 20.0,
    #     "iterations": 20,
    # },
    # {
    #     "factors": 128,
    #     "regularization": 0.10,
    #     "alpha": 40.0,
    #     "iterations": 20,
    # },
]



In [57]:
validation_trials = []

for trial_number, params in enumerate(
    PARAM_GRID,
    start=1,
):
    print()
    print(
        f"Trial {trial_number}/{len(PARAM_GRID)}"
    )
    print(params)

    als = ALSCandidateModel(
        **params,
        random_state=RANDOM_STATE,
        num_threads=0,
    )

    fit_started = time.perf_counter()

    als.fit(
        interactions=train_history,
        user_col="user_id",
        item_col="item_id",
        show_progress=True,
    )

    fit_seconds = (
        time.perf_counter() - fit_started
    )

    evaluation_started = time.perf_counter()

    validation_metrics = evaluate_recall_at_k(
        model=als,
        targets=validation_targets,
        k=K,
        batch_size=BATCH_SIZE,
    )

    evaluation_seconds = (
        time.perf_counter()
        - evaluation_started
    )

    validation_trials.append(
        {
            "params": params.copy(),
            "metrics": validation_metrics.copy(),
            "fit_seconds": fit_seconds,
            "evaluation_seconds": evaluation_seconds,
        }
    )

    print(
        f"Recall@{K} warm: "
        f"{validation_metrics[f'recall@{K}_warm']:.5f}"
    )

    print(
        f"Recall@{K} all:  "
        f"{validation_metrics[f'recall@{K}_all']:.5f}"
    )

    print(
        "Warm users:      "
        f"{validation_metrics['warm_user_share']:.2%}"
    )

    print(
        "Достижимые positives: "
        f"{validation_metrics['retrievable_positive_share']:.2%}"
    )

    print(
        f"Обучение: {fit_seconds:.1f} секунд"
    )

    print(
        "Оценка:   "
        f"{evaluation_seconds:.1f} секунд"
    )

    del als
    gc.collect()


Trial 1/1
{'factors': 32, 'regularization': 0.05, 'alpha': 10.0, 'iterations': 20}


  0%|          | 0/20 [00:00<?, ?it/s]

Recall@200:   0%|          | 0/6 [00:00<?, ?it/s]

Recall@200 warm: 0.33576
Recall@200 all:  0.30142
Warm users:      100.00%
Достижимые positives: 88.16%
Обучение: 1.5 секунд
Оценка:   3.2 секунд


In [58]:
validation_results = pd.DataFrame(
    [
        {
            **trial["params"],
            **trial["metrics"],
            "fit_seconds": trial["fit_seconds"],
            "evaluation_seconds": (
                trial["evaluation_seconds"]
            ),
        }
        for trial in validation_trials
    ]
)

validation_results = (
    validation_results
    .sort_values(
        by=f"recall@{K}_warm",
        ascending=False,
    )
    .reset_index(drop=True)
)

display(validation_results)

,factors,regularization,alpha,iterations,recall@200_all,recall@200_warm,warm_user_share,retrievable_positive_share,target_users,warm_target_users,evaluated_warm_users,target_positives,retrievable_positives,fit_seconds,evaluation_seconds
0,32,0.05,10.0,20,0.301422,0.335761,1.0,0.881568,11615,11615,10954,28185,24847,1.547419,3.194662


In [59]:
best_trial = max(
    validation_trials,
    key=lambda trial: trial["metrics"][
        f"recall@{K}_warm"
    ],
)

best_params = best_trial["params"].copy()
best_validation_metrics = (
    best_trial["metrics"].copy()
)

print("Лучшие параметры:")
print(best_params)

print()
print(
    f"Validation Recall@{K} warm: "
    f"{best_validation_metrics[f'recall@{K}_warm']:.5f}"
)

print(
    f"Validation Recall@{K} all:  "
    f"{best_validation_metrics[f'recall@{K}_all']:.5f}"
)

Лучшие параметры:
{'factors': 32, 'regularization': 0.05, 'alpha': 10.0, 'iterations': 20}

Validation Recall@200 warm: 0.33576
Validation Recall@200 all:  0.30142


In [60]:
final_history = (
    pd.concat(
        [
            train[["user_id", "item_id"]],
            validation[["user_id", "item_id"]],
        ],
        ignore_index=True,
    )
    .drop_duplicates()
    .reset_index(drop=True)
)

print(
    f"Финальная история: {len(final_history):,}"
)

Финальная история: 152,528


### Обучение финальной модели на train + validation и проверка на test

In [61]:
final_als = ALSCandidateModel(
    **best_params,
    random_state=RANDOM_STATE,
    num_threads=0,
)

final_fit_started = time.perf_counter()

final_als.fit(
    interactions=final_history,
    user_col="user_id",
    item_col="item_id",
    show_progress=True,
)

final_fit_seconds = (
    time.perf_counter() - final_fit_started
)

print(
    f"Обучение заняло "
    f"{final_fit_seconds:.1f} секунд"
)

  0%|          | 0/20 [00:00<?, ?it/s]

Обучение заняло 1.7 секунд


In [62]:
test_metrics = evaluate_recall_at_k(
    model=final_als,
    targets=test_targets,
    k=K,
    batch_size=BATCH_SIZE,
)
NDCG_K = 10
test_ndcg_10 = evaluate_ndcg_at_k(
    model=final_als,
    targets=test_targets,
    k=NDCG_K,
    batch_size=BATCH_SIZE,
)

Recall@200:   0%|          | 0/4 [00:00<?, ?it/s]

NDCG@10:   0%|          | 0/4 [00:00<?, ?it/s]

In [63]:
print(
    f"Test Recall@{K}, warm: "
    f"{test_metrics[f'recall@{K}_warm']:.5f}"
)

print(
    f"Test Recall@{K}, all:  "
    f"{test_metrics[f'recall@{K}_all']:.5f}"
)

print(
    f"Test NDCG@{NDCG_K}, warm: "
    f"{test_ndcg_10[f'ndcg@{NDCG_K}_warm']:.5f}"
)

print(
    f"Test NDCG@{NDCG_K}, all:  "
    f"{test_ndcg_10[f'ndcg@{NDCG_K}_all']:.5f}"
)


print(
    "Доля warm-пользователей: "
    f"{test_metrics['warm_user_share']:.2%}"
)

print(
    "Доля достижимых positives (которые были в train или validation): "
    f"{test_metrics['retrievable_positive_share']:.2%}"
)

Test Recall@200, warm: 0.31623
Test Recall@200, all:  0.28349
Test NDCG@10, warm: 0.03515
Test NDCG@10, all:  0.03175
Доля warm-пользователей: 100.00%
Доля достижимых positives (которые были в train или validation): 88.81%


#### Работа с cold-start 
В текущем эксперименте модели сравниваются только на тёплых пользователях — пользователях, которые имеют хотя бы одно взаимодействие в обучающей истории. Холодные пользователи исключаются из оценки, обработка cold-start остаётся за рамками текущего исследования.

посчитаем максимально возможный NDCG@10, который можно получить среди отобранных 200 фильмов

In [64]:
test_user_ids = (
    test_targets["user_id"]
    .drop_duplicates()
    .tolist()
)

test_candidates_200 = final_als.recommend(
    user_ids=test_user_ids,
    n=200,
    filter_seen=True,
    batch_size=BATCH_SIZE,
)

oracle_ndcg_10_all = oracle_ndcg_at_k(
    candidates=test_candidates_200,
    targets=test_targets,
    k=10,
)

print(
    f"Oracle NDCG@10 all: "
    f"{oracle_ndcg_10_all:.5f}"
)

Oracle NDCG@10 all: 0.31449


In [65]:
known_items = set(
    final_als.item_ids.tolist()
)

warm_test_targets = (
    test_targets[
        test_targets["item_id"].isin(known_items)
    ]
    .copy()
)

oracle_ndcg_10_warm = oracle_ndcg_at_k(
    candidates=test_candidates_200,
    targets=warm_test_targets,
    k=10,
)

print(
    f"Oracle NDCG@10 warm: "
    f"{oracle_ndcg_10_warm:.5f}"
)

Oracle NDCG@10 warm: 0.34677


После этапаа кандидатогенерации имеем следующие результаты:

#

In [66]:
als_summary = pd.DataFrame(
    [
        {
            "Срез": "All",
            "Recall@200": test_metrics[f"recall@{K}_all"],
            "NDCG@10": test_ndcg_10[f"ndcg@{NDCG_K}_all"],
            "Oracle NDCG@10": oracle_ndcg_10_all,
            "Доля от oracle": (
                test_ndcg_10[f"ndcg@{NDCG_K}_all"]
                / oracle_ndcg_10_all
            ),
            "Доля пользователей": 1.0,
            "Доля positives": 1.0,
        },
        {
            "Срез": "Warm",
            "Recall@200": test_metrics[f"recall@{K}_warm"],
            "NDCG@10": test_ndcg_10[f"ndcg@{NDCG_K}_warm"],
            "Oracle NDCG@10": oracle_ndcg_10_warm,
            "Доля от oracle": (
                test_ndcg_10[f"ndcg@{NDCG_K}_warm"]
                / oracle_ndcg_10_warm
            ),
            "Доля пользователей": test_metrics["warm_user_share"],
            "Доля positives": test_metrics[
                "retrievable_positive_share"
            ],
        },
    ]
)

display(
    als_summary.style.format(
        {
            "Recall@200": "{:.5f}",
            "NDCG@10": "{:.5f}",
            "Oracle NDCG@10": "{:.5f}",
            "Доля от oracle": "{:.2%}",
            "Доля пользователей": "{:.2%}",
            "Доля positives": "{:.2%}",
        }
    )
)

,Срез,Recall@200,NDCG@10,Oracle NDCG@10,Доля от oracle,Доля пользователей,Доля positives
0,All,0.28349,0.03175,0.31449,10.10%,100.00%,100.00%
1,Warm,0.31623,0.03515,0.34677,10.14%,100.00%,88.81%


Следующая задача - улучшить ранжирование внутри 200 выбранных фильмов

## Catboost

In [67]:
from catboost import CatBoostRanker, Pool
from src import build_manual_features

In [68]:
users = pd.read_csv(
    PROJECT_ROOT / "data" / "users.csv"
)

items = pd.read_csv(
    PROJECT_ROOT / "data" / "items.csv"
)

отрежем часть train для опредения таргетов

In [69]:
RANKER_TRAIN_CUTOFF = "2021-06-01"

train_dates = pd.to_datetime(
    train["last_watch_dt"]
)

ranker_history = (
    train[
        train_dates
        < pd.Timestamp(RANKER_TRAIN_CUTOFF)
    ]
    .copy()
    .reset_index(drop=True)
)

ranker_train_targets = (
    train[
        train_dates
        >= pd.Timestamp(RANKER_TRAIN_CUTOFF)
    ][["user_id", "item_id"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

print(
    f"История для CatBoost train: "
    f"{len(ranker_history):,}"
)

print(
    f"Положительных взаимодействий: "
    f"{len(ranker_train_targets):,}"
)

История для CatBoost train: 69,923
Положительных взаимодействий: 54,420


In [70]:
def add_candidate_labels(
    candidates: pd.DataFrame,
    targets: pd.DataFrame,
) -> pd.DataFrame:
    positive_pairs = (
        targets[["user_id", "item_id"]]
        .drop_duplicates()
        .assign(label=1)
    )

    labeled_candidates = candidates.merge(
        positive_pairs,
        on=["user_id", "item_id"],
        how="left",
        validate="one_to_one",
    )

    labeled_candidates["label"] = (
        labeled_candidates["label"]
        .fillna(0)
        .astype("int8")
    )

    return labeled_candidates

In [71]:
ranker_train_als = ALSCandidateModel(
    **best_params,
    random_state=RANDOM_STATE,
    num_threads=0,
)

ranker_train_als.fit(
    interactions=ranker_history[
        ["user_id", "item_id"]
    ],
    show_progress=True,
)

  0%|          | 0/20 [00:00<?, ?it/s]

In [72]:
ranker_train_user_ids = (
    ranker_train_targets["user_id"]
    .drop_duplicates()
    .tolist()
)

ranker_train_candidates = (
    ranker_train_als.recommend(
        user_ids=ranker_train_user_ids,
        n=K,
        filter_seen=True,
        batch_size=BATCH_SIZE,
    )
)

ranker_train_candidates = add_candidate_labels(
    candidates=ranker_train_candidates,
    targets=ranker_train_targets,
)

In [73]:
print(
    f"Кандидатов: "
    f"{len(ranker_train_candidates):,}"
)

print(
    f"Положительных кандидатов: "
    f"{ranker_train_candidates['label'].sum():,}"
)

print(
    "Доля положительных строк: "
    f"{ranker_train_candidates['label'].mean():.4%}"
)

Кандидатов: 1,504,400
Положительных кандидатов: 4,366
Доля положительных строк: 0.2902%


In [74]:
positive_train_users = (
    ranker_train_candidates
    .groupby("user_id")["label"]
    .max()
)

positive_train_users = (
    positive_train_users[
        positive_train_users == 1
    ]
    .index
)

ranker_train_candidates = (
    ranker_train_candidates[
        ranker_train_candidates["user_id"].isin(
            positive_train_users
        )
    ]
    .sort_values(["user_id", "als_rank"])
    .reset_index(drop=True)
)

print(
    "Пользователей с найденным positive: "
    f"{len(positive_train_users):,}"
)

Пользователей с найденным positive: 3,179


уберем пользователей у которых не нашлось ни одного позитива

In [75]:
X_train, categorical_columns = (
    build_manual_features(
        candidates=ranker_train_candidates.drop(
            columns="label"
        ),
        history=ranker_history,
        users=users,
        items=items,
        cutoff=RANKER_TRAIN_CUTOFF,
    )
)

y_train = (
    ranker_train_candidates["label"]
    .to_numpy()
)

train_group_ids = (
    ranker_train_candidates["user_id"]
    .to_numpy()
)

print(f"Train shape: {X_train.shape}")
print(f"Categorical: {categorical_columns}")

Train shape: (635800, 48)
Categorical: ['user_id', 'item_id', 'age', 'income', 'sex', 'kids_flg', 'content_type', 'for_kids', 'item_primary_genre', 'item_primary_country', 'user_top_genre', 'user_top_country', 'user_top_content_type']


In [76]:
validation_als = ALSCandidateModel(
    **best_params,
    random_state=RANDOM_STATE,
    num_threads=0,
)

validation_als.fit(
    interactions=train[
        ["user_id", "item_id"]
    ],
    show_progress=True,
)

  0%|          | 0/20 [00:00<?, ?it/s]

In [77]:
validation_user_ids = (
    validation_targets["user_id"]
    .drop_duplicates()
    .tolist()
)

validation_candidates = (
    validation_als.recommend(
        user_ids=validation_user_ids,
        n=K,
        filter_seen=True,
        batch_size=BATCH_SIZE,
    )
)

validation_candidates = add_candidate_labels(
    candidates=validation_candidates,
    targets=validation_targets,
)

validation_candidates = (
    validation_candidates
    .sort_values(["user_id", "als_rank"])
    .reset_index(drop=True)
)

print(
    f"Validation candidates: "
    f"{len(validation_candidates):,}"
)

print(
    f"Validation positives: "
    f"{validation_candidates['label'].sum():,}"
)

Validation candidates: 2,323,000
Validation positives: 7,522


In [78]:
X_validation, _ = build_manual_features(
    candidates=validation_candidates.drop(
        columns="label"
    ),
    history=train,
    users=users,
    items=items,
    cutoff=TRAIN_CUTOFF,
)


X_validation = X_validation[
    X_train.columns
]

y_validation = (
    validation_candidates["label"]
    .to_numpy()
)

validation_group_ids = (
    validation_candidates["user_id"]
    .to_numpy()
)

In [79]:
train_pool = Pool(
    data=X_train,
    label=y_train,
    group_id=train_group_ids,
    cat_features=categorical_columns,
)

validation_pool = Pool(
    data=X_validation,
    label=y_validation,
    group_id=validation_group_ids,
    cat_features=categorical_columns,
)

In [80]:
catboost_ranker = CatBoostRanker(
    loss_function="YetiRank",
    eval_metric="NDCG:top=10",
    iterations=1_000,
    learning_rate=0.05,
    depth=8,
    l2_leaf_reg=5.0,
    random_seed=RANDOM_STATE,
    verbose=100,
    allow_writing_files=False,
)

catboost_ranker.fit(
    train_pool,
    eval_set=validation_pool,
    early_stopping_rounds=100,
    use_best_model=True,
)

Groupwise loss function. OneHotMaxSize set to 10
0:	test: 0.5845959	best: 0.5845959 (0)	total: 1.84s	remaining: 30m 39s


KeyboardInterrupt: 

In [ ]:
print(
    "Лучшая итерация:",
    catboost_ranker.get_best_iteration(),
)

print(
    "Лучшие validation-метрики:",
    catboost_ranker.get_best_score(),
)

Лучшая итерация: 65
Лучшие validation-метрики: {}


In [ ]:
validation_scores = (
    catboost_ranker.predict(X_validation)
)

scored_validation_candidates = (
    validation_candidates[
        ["user_id", "item_id"]
    ]
    .copy()
)

scored_validation_candidates["score"] = (
    validation_scores
)

In [ ]:
def ndcg_at_k_from_scores(
    scored_candidates: pd.DataFrame,
    targets: pd.DataFrame,
    k: int = 10,
) -> float:
    actual_by_user = (
        targets[["user_id", "item_id"]]
        .drop_duplicates()
        .groupby("user_id")["item_id"]
        .agg(set)
        .to_dict()
    )

    ranked_by_user = (
        scored_candidates
        .sort_values(
            ["user_id", "score"],
            ascending=[True, False],
        )
        .groupby("user_id")["item_id"]
        .agg(list)
        .to_dict()
    )

    user_ndcg = []

    for user_id, actual_items in actual_by_user.items():
        ranked_items = ranked_by_user.get(
            user_id,
            [],
        )[:k]

        relevance = np.array(
            [
                item_id in actual_items
                for item_id in ranked_items
            ],
            dtype=float,
        )

        discounts = np.log2(
            np.arange(2, len(ranked_items) + 2)
        )

        dcg = np.sum(
            relevance / discounts
        )

        ideal_length = min(
            len(actual_items),
            k,
        )

        idcg = np.sum(
            1 / np.log2(
                np.arange(2, ideal_length + 2)
            )
        )

        user_ndcg.append(
            dcg / idcg if idcg > 0 else 0.0
        )

    return float(np.mean(user_ndcg))

In [ ]:
catboost_validation_ndcg_10 = (
    ndcg_at_k_from_scores(
        scored_candidates=(
            scored_validation_candidates
        ),
        targets=validation_targets,
        k=10,
    )
)

print(
    "CatBoost validation NDCG@10: "
    f"{catboost_validation_ndcg_10:.5f}"
)

CatBoost validation NDCG@10: 0.08246


In [ ]:
positive_validation_users = (
    validation_candidates
    .groupby("user_id")["label"]
    .max()
)

positive_validation_users = (
    positive_validation_users[
        positive_validation_users == 1
    ]
    .index
)

validation_train_mask = (
    validation_candidates["user_id"]
    .isin(positive_validation_users)
)

X_validation_for_train = (
    X_validation.loc[validation_train_mask]
    .reset_index(drop=True)
)

y_validation_for_train = (
    validation_candidates.loc[
        validation_train_mask,
        "label",
    ]
    .to_numpy()
)

validation_users_for_train = (
    validation_candidates.loc[
        validation_train_mask,
        "user_id",
    ]
    .reset_index(drop=True)
)

print(
    "Validation-групп для финального обучения:",
    len(positive_validation_users),
)

Validation-групп для финального обучения: 990


In [ ]:
X_final_train = pd.concat(
    [
        X_train.reset_index(drop=True),
        X_validation_for_train,
    ],
    ignore_index=True,
)

y_final_train = np.concatenate(
    [
        y_train,
        y_validation_for_train,
    ]
)

train_group_keys = (
    "june_"
    + ranker_train_candidates["user_id"]
    .astype(str)
    .reset_index(drop=True)
)

validation_group_keys = (
    "july_"
    + validation_users_for_train.astype(str)
)

final_group_keys = pd.concat(
    [
        train_group_keys,
        validation_group_keys,
    ],
    ignore_index=True,
)

final_group_ids = pd.factorize(
    final_group_keys,
    sort=False,
)[0]

print("Финальный train:", X_final_train.shape)
print("Число групп:", len(np.unique(final_group_ids)))
print("Число positives:", int(y_final_train.sum()))

Финальный train: (308400, 48)
Число групп: 1542
Число positives: 2049


In [ ]:
best_iterations = (
    catboost_ranker.get_best_iteration() + 1
)

if best_iterations <= 0:
    best_iterations = 1_000

print("Количество итераций:", best_iterations)

Количество итераций: 66


In [ ]:
final_train_pool = Pool(
    data=X_final_train,
    label=y_final_train,
    group_id=final_group_ids,
    cat_features=categorical_columns,
)

final_catboost_ranker = CatBoostRanker(
    loss_function="YetiRank",
    eval_metric="NDCG:top=10",
    iterations=best_iterations,
    learning_rate=0.05,
    depth=8,
    l2_leaf_reg=5.0,
    random_seed=RANDOM_STATE,
    verbose=100,
    allow_writing_files=False,
)

final_catboost_ranker.fit(
    final_train_pool,
)

Groupwise loss function. OneHotMaxSize set to 10
0:	total: 355ms	remaining: 23s
65:	total: 17.2s	remaining: 0us


CatBoostRanker(allow_writing_files=False, depth=8, eval_metric='NDCG:top=10', iterations=66, l2_leaf_reg=5.0, learning_rate=0.05, loss_function='YetiRank', random_seed=42, verbose=100)

In [ ]:
test_candidates = (
    test_candidates_200
    .sort_values(["user_id", "als_rank"])
    .reset_index(drop=True)
)

In [ ]:
test_history = (
    pd.concat(
        [
            train,
            validation,
        ],
        ignore_index=True,
    )
    .reset_index(drop=True)
)

X_test, _ = build_manual_features(
    candidates=test_candidates,
    history=test_history,
    users=users,
    items=items,
    cutoff=VALIDATION_CUTOFF,
)

X_test = X_test[X_final_train.columns]

print("Test shape:", X_test.shape)

Test shape: (278200, 48)


In [ ]:
test_scores = final_catboost_ranker.predict(
    X_test
)

scored_test_candidates = (
    test_candidates[
        ["user_id", "item_id"]
    ]
    .copy()
)

scored_test_candidates["score"] = test_scores

scored_test_candidates = (
    scored_test_candidates
    .sort_values(
        ["user_id", "score"],
        ascending=[True, False],
    )
    .reset_index(drop=True)
)

scored_test_candidates["catboost_rank"] = (
    scored_test_candidates
    .groupby("user_id")
    .cumcount()
    + 1
)

scored_test_candidates.head(10)

,user_id,item_id,score,catboost_rank
0,1000,15297,0.776932,1
1,1000,12192,0.552712,2
2,1000,16228,0.472459,3
3,1000,7829,0.444760,4
4,1000,4495,0.412139,5
5,1000,6192,0.312090,6
6,1000,2720,0.307275,7
7,1000,8636,0.294523,8
8,1000,6809,0.281683,9
9,1000,5693,0.259860,10


In [ ]:
catboost_test_ndcg_10 = ndcg_at_k_from_scores(
    scored_candidates=scored_test_candidates,
    targets=test_targets,
    k=10,
)

print(
    "CatBoost Test NDCG@10: "
    f"{catboost_test_ndcg_10:.5f}"
)

CatBoost Test NDCG@10: 0.06409


In [ ]:
comparison = pd.DataFrame(
    {
        "Модель": [
            "ALS",
            "ALS + CatBoost",
            
        ],
        "Test NDCG@10": [
            test_ndcg_10["ndcg@10_all"],
            catboost_test_ndcg_10,
            
        ],
    }
)

comparison["Прирост относительно ALS"] = (
    comparison["Test NDCG@10"]
    / test_ndcg_10["ndcg@10_all"]
    - 1
)

display(
    comparison.style.format(
        {
            "Test NDCG@10": "{:.5f}",
            "Прирост относительно ALS": "{:+.2%}",
        }
    )
)

,Модель,Test NDCG@10,Прирост относительно ALS
0,ALS,0.03081,+0.00%
1,ALS + CatBoost,0.06409,+108.05%
